# HU3 — Entrenamiento del modelo con partidas de referencia

Corre en Google Colab (GPU gratis) los pasos 3-5 de la sección 16 de
`PLAN_IMPLEMENTACION_COMPLETO.md`:

1. Probar el pipeline con un subconjunto chico (100-200 partidas) antes de escalar.
2. Entrenar una primera versión simple del modelo — el objetivo es que el pipeline
   funcione de punta a punta, no lograr precisión alta todavía.
3. Guardar el checkpoint en Google Drive, no solo en la sesión de Colab (se pierde
   al cerrarla).

Activar GPU antes de correr: `Entorno de ejecución > Cambiar tipo de entorno de
ejecución > GPU`.

Contrato de datos: tablero -> tensor (8, 8, 12) en perspectiva del jugador a mover,
jugada -> clase `origen*64 + destino` (4096 clases). Ver `training/data_pipeline.py`
y `backend/servicios/aprendizaje/modelo_jugadas.py`.

## 1. Traer el código del repositorio

Solo el código (`training/`, `backend/`) — el dataset de partidas no está en git
(pesa 1.7 GB, ver `.gitignore`), se baja aparte en el paso 3.

In [ ]:
REPO_URL = "https://github.com/larzekao9/Brazo-Rob-tico-con-Inteligencia-Artificial-para-el-Aprendizaje-del-Ajedrez.git"

!rm -rf /content/ajedrez
!git clone --depth 1 "$REPO_URL" /content/ajedrez
%cd /content/ajedrez

Si el clone falla con `repository not found` es porque el repo es privado. En ese
caso, generar un token de acceso personal en GitHub (Settings > Developer settings >
Personal access tokens, permiso `repo` de solo lectura) y correr esta celda en vez
de la de arriba. El token se pide de forma interactiva (`getpass`) y no queda
guardado en ningún archivo del notebook, tal como pide `CLAUDE.md`.

In [ ]:
# Alternativa si el repo es privado (dejar comentado si el clone de arriba funcionó):

# from getpass import getpass
# usuario_github = getpass("Usuario de GitHub: ")
# token_github = getpass("Token de acceso personal (no se guarda en el notebook): ")
# url_autenticada = REPO_URL.replace("https://", f"https://{usuario_github}:{token_github}@")
# !rm -rf /content/ajedrez
# !git clone --depth 1 "$url_autenticada" /content/ajedrez
# %cd /content/ajedrez
# del token_github, url_autenticada

## 2. Dependencias

`torch` y `numpy` ya vienen instalados en el entorno de Colab. Solo faltan
`python-chess` y `zstandard`, con las mismas versiones fijadas en `requirements.txt`
para que el pipeline se comporte igual acá que en la máquina de cada uno.

In [ ]:
!pip install -q python-chess==1.999.0 zstandard==0.25.0
!apt-get update -qq && apt-get install -y -qq stockfish
!ln -sf /usr/games/stockfish /usr/local/bin/stockfish


## 3. Bajar el mes de partidas de Lichess

Mismo archivo que ya se usó para probar el pipeline localmente (2017-02, ~1.7 GB
comprimido — el mes más liviano disponible, ver `docs/plan_sprints.md`, HU3). El
pipeline lee en streaming y corta apenas junta `LIMITE_PARTIDAS`, así que no hace
falta esperar a que termine de descomprimirse para empezar a entrenar con el
subconjunto chico — pero si el subconjunto quedara al principio del archivo, la
descarga completa igual da margen para escalar sin volver a bajar nada (HU4).

In [ ]:
!mkdir -p training/data
!wget -q --show-progress -O training/data/lichess_db_standard_rated_2017-02.pgn.zst \
    https://database.lichess.org/standard/lichess_db_standard_rated_2017-02.pgn.zst
!ls -lh training/data/

## 4. Montar Google Drive

Para que el checkpoint del paso 7 sobreviva al cierre de la sesión de Colab (una
sesión de Colab se borra sola; Drive no).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

CARPETA_CHECKPOINTS_DRIVE = "/content/drive/MyDrive/ajedrez_checkpoints"
import os
os.makedirs(CARPETA_CHECKPOINTS_DRIVE, exist_ok=True)

## 5. Generar el subconjunto chico (100-200 partidas)

Usa el mismo `pgn_to_samples` que ya está probado con `training/test_data_pipeline.py`
— nada de lógica nueva acá, solo se corre a mayor escala que en la máquina local.

In [ ]:
import sys
sys.path.insert(0, '/content/ajedrez')

from training.data_pipeline import pgn_to_samples

# HU4: Filtrar por ELO mínimo para entrenar solo con partidas de maestros y jugadores fuertes
LIMITE_PARTIDAS = 25000  # 3000 partidas de maestros dan ~200,000 posiciones de alta calidad
ELO_MINIMO = 2000

print(f'Extrayendo {LIMITE_PARTIDAS} partidas de maestros con ELO >= {ELO_MINIMO}...')
muestras = pgn_to_samples(
    'training/data/lichess_db_standard_rated_2017-02.pgn.zst',
    limite_partidas=LIMITE_PARTIDAS,
    elo_minimo=ELO_MINIMO,
)
print(f'¡Listo! {len(muestras)} muestras (posición, jugada) de {LIMITE_PARTIDAS} partidas de nivel maestro')


## 6. Dataset y split entrenamiento/validación

Split simple 90/10 al azar — alcanza para esta primera versión (HU3); una
evaluación más rigurosa (por partidas, no por posición suelta, para no filtrar
información entre train y val) queda para HU4.

In [ ]:
import random

import torch
from torch.utils.data import DataLoader, Dataset

from backend.servicios.aprendizaje.modelo_jugadas import (
    NUM_CLASES,
    RedPrediccionJugadas, RedResNetAjedrez, RedSEResNetAjedrez,
    tensor_a_entrada_red,
)


class DatasetJugadas(Dataset):
    def __init__(self, muestras):
        self.muestras = muestras

    def __len__(self):
        return len(self.muestras)

    def __getitem__(self, indice):
        tensor_posicion, etiqueta = self.muestras[indice]
        return tensor_a_entrada_red(tensor_posicion), etiqueta


random.seed(0)
muestras_mezcladas = muestras[:]
random.shuffle(muestras_mezcladas)
corte = int(0.9 * len(muestras_mezcladas))
muestras_train = muestras_mezcladas[:corte]
muestras_val = muestras_mezcladas[corte:]

cargador_train = DataLoader(DatasetJugadas(muestras_train), batch_size=128, shuffle=True)
cargador_val = DataLoader(DatasetJugadas(muestras_val), batch_size=128)

print(f"train: {len(muestras_train)} — val: {len(muestras_val)}")

## 7. Entrenar

Pocas épocas, a propósito — esta corrida es para validar que el pipeline funciona
de punta a punta, no para maximizar accuracy (eso es HU4, con el mes completo).

In [ ]:
dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
print('dispositivo:', dispositivo)

# Red ResNet con 4 bloques residuales (v3) para ajedrez
# Red SE-ResNet con 6 bloques y atencion por canales Squeeze-and-Excitation (v5 - Maestria)
red = RedSEResNetAjedrez(canales=192, cantidad_bloques=8).to(dispositivo)
optimizador = torch.optim.AdamW(red.parameters(), lr=1e-3, weight_decay=1e-4)
CANTIDAD_EPOCAS = 25
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizador, T_max=CANTIDAD_EPOCAS)
funcion_perdida = torch.nn.CrossEntropyLoss(label_smoothing=0.05)


def evaluar(cargador):
    red.eval()
    correctas, total = 0, 0
    with torch.no_grad():
        for entradas, etiquetas in cargador:
            entradas, etiquetas = entradas.to(dispositivo), etiquetas.to(dispositivo)
            predicciones = red(entradas).argmax(dim=1)
            correctas += (predicciones == etiquetas).sum().item()
            total += etiquetas.size(0)
    return correctas / total if total else 0.0


for epoca in range(1, CANTIDAD_EPOCAS + 1):
    red.train()
    perdida_acumulada = 0.0
    for entradas, etiquetas in cargador_train:
        entradas, etiquetas = entradas.to(dispositivo), etiquetas.to(dispositivo)
        optimizador.zero_grad()
        salida = red(entradas)
        perdida = funcion_perdida(salida, etiquetas)
        perdida.backward()
        optimizador.step()
        perdida_acumulada += perdida.item() * entradas.size(0)

    scheduler.step()
    perdida_promedio = perdida_acumulada / len(muestras_train)
    accuracy_val = evaluar(cargador_val)
    print(f'época {epoca}/{CANTIDAD_EPOCAS} — pérdida train: {perdida_promedio:.4f} — accuracy val: {accuracy_val:.2%}')


Con solo 200 partidas, un accuracy val bajo (muy por debajo de un modelo tipo Maia)
es esperable y no es un error — recién con el mes completo (HU4) tiene sentido
empezar a juzgar la calidad real del modelo. Lo que esta corrida valida es que el
pipeline completo (PGN -> tensores -> entrenamiento -> checkpoint) funciona sin
errores de punta a punta.

## 8. Guardar el checkpoint en Google Drive

In [ ]:
import datetime

nombre_checkpoint = f'modelo_jugadas_v5_{datetime.date.today().isoformat()}.pt'
ruta_checkpoint = f'{CARPETA_CHECKPOINTS_DRIVE}/{nombre_checkpoint}'

torch.save(
    {
        'state_dict': red.state_dict(),
        'num_clases': NUM_CLASES,
        'arquitectura': 'se_resnet',
        'canales': 192,
        'cantidad_bloques': 8,
        'cantidad_partidas': LIMITE_PARTIDAS,
        'elo_minimo': ELO_MINIMO,
        'cantidad_muestras': len(muestras),
        'accuracy_val': accuracy_val,
        'fecha': datetime.date.today().isoformat(),
    },
    ruta_checkpoint,
)
print('guardado en', ruta_checkpoint)


## Próximos pasos (HU4, no de este notebook)

- Subir `LIMITE_PARTIDAS` al mes completo una vez que esta corrida chica funcione
  sin errores.
- Versionar los checkpoints (no sobreescribir siempre el mismo nombre en Drive —
  esta celda ya arma un nombre por fecha, pero falta un criterio de promoción).
- Evaluar cada versión candidata antes de promoverla (RF15) — todavía no hay
  ninguna métrica más allá de accuracy de validación.
- `backend/servicios/aprendizaje/inferencia.py` — cargar este checkpoint y
  exponerlo detrás de la interfaz `EstrategiaJugada` (`EstrategiaModelo`, ver
  sección 4.1 de `PLAN_IMPLEMENTACION_COMPLETO.md`).

## 9. Evaluación científica del modelo v3

Mide el Accuracy Top-1 y la pérdida en centipawns contra partidas humanas nunca vistas,
usando Stockfish como oráculo de comparación (HU4).

In [ ]:
from training.evaluar_modelo import evaluar_modelo, BALDES_ERROR

print('--- Evaluando Modelo v5 (SE-ResNet-8 192C + 25k Partidas Maestras) ---')
resultado = evaluar_modelo(
    'training/data/lichess_db_standard_rated_2017-02.pgn.zst',
    cantidad_partidas=20,
    saltar_partidas=35000,  # Partidas no vistas
    ruta_checkpoint=ruta_checkpoint,
)

print(f'\nTotal jugadas evaluadas: {resultado.total_jugadas}')
print(f'Accuracy Top-1: {resultado.accuracy:.2%}')
print('Distribución de errores (contra Stockfish):')
for balde in BALDES_ERROR:
    print(f'  {balde}: {resultado.distribucion_errores[balde]}')
